# Lorenz attractor and the butterfly effect

*others · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/lorenz_attractor)

Three coupled equations from a model of convection never settle and never repeat. Written as a linear part $A_{ij}x_j$ plus a quadratic part $Q_{ijk}x_j x_k$ with a rank-3 tensor, one einsum moves a whole batch of trajectories that start within $10^{-6}$ of each other; they peel apart exponentially, at the rate set by the largest Lyapunov exponent $\lambda \approx 0.9$, until they are spread over the whole butterfly.

**Vector form**

$$\dot x = \sigma\,(y - x),\qquad \dot y = x\,(\rho - z) - y,\qquad \dot z = x\,y - \beta\,z$$

**Index (tensor) form**

$$\dot x_{bi} = A_{ij}\,x_{bj} + Q_{ijk}\,x_{bj}\,x_{bk},\qquad Q_{102} = -1,\quad Q_{201} = 1,\qquad \partial_i \dot x_i = A_{ii} + (Q_{iik} + Q_{iki})\,x_k = -(\sigma + 1 + \beta)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bi}$ | `x` | (B, 3) | state $(x, y, z)$ of trajectory $b$ |
| $A_{ij}$ | `lorenz_tensors(sigma, rho, beta)[0]` | (3, 3) | linear part: rows $(-\sigma, \sigma, 0)$, $(\rho, -1, 0)$, $(0, 0, -\beta)$ |
| $Q_{ijk}$ | `lorenz_tensors(sigma, rho, beta)[1]` | (3, 3, 3) | quadratic part, a rank-3 tensor with two non-zero entries: $-xz$ in $\dot y$ and $+xy$ in $\dot z$ |
| $A_{ij}x_{bj}$ | `einsum('ij,bj->bi', A, x)` | (B, 3) | repeated $j$ ⇒ summed |
| $Q_{ijk}x_{bj}x_{bk}$ | `einsum('ijk,bj,bk->bi', Q, x, x)` | (B, 3) | repeated $j$ and $k$ ⇒ summed: the nonlinearity that makes the chaos |
| $\partial_i \dot x_i$ | `divergence(x, A, Q)` | (B,) | trace of the Jacobian: volumes of states shrink at the constant rate $\sigma + 1 + \beta$ |
| $s(t)$ | `spread(traj)` | (T,) | RMS distance of the trajectories from their mean |
| $\lambda$ | `lyapunov_fit(times, s)` | () | slope of $\ln s$ vs $t$ while the spread grows from $10\,s(0)$ to $1$ |

Lorenz (1963) cut a model of a fluid heated from below down to three numbers: $x$ is the strength of the convection roll, $y$ and $z$ describe the temperature profile. With $\sigma = 10$, $\rho = 28$ and $\beta = 8/3$ the motion circles one of the two fixed points $C_\pm = (\pm\sqrt{\beta(\rho-1)}, \pm\sqrt{\beta(\rho-1)}, \rho-1)$, then jumps unpredictably to the other wing. Nearby trajectories separate like $e^{\lambda t}$ with $\lambda \approx 0.906$: an error of $10^{-6}$ grows to the size of the attractor after about $15$ time units, which is why a forecast has a horizon (the butterfly effect). At the same time every volume of states shrinks like $e^{-(\sigma+1+\beta)t}$, so the trajectories crowd onto a thin, fractal set: the strange attractor. The tensor form makes the structure explicit: $A$ holds all the linear couplings and the whole nonlinearity is the two entries of $Q$. Before the run, one trajectory from $(1, 1, 1)$ is integrated for $10$ time units so that the ball starts on the attractor; RK4 with a fixed step $0.005$ is accurate enough here. Try $\rho < 1$: the origin becomes stable and every trajectory decays to it; for $1 < \rho < 24.74$ they spiral into $C_+$ or $C_-$; near $\rho = 28$ and above the chaos returns.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/lorenz_attractor/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    sigma, rho, beta = params["sigma"], params["rho"], params["beta"]
    A, Q = tensors.lorenz_tensors(sigma, rho, beta, dev)  # (3, 3), (3, 3, 3)

    center = solve.settle(A, Q)  # a point on the attractor
    radius = 10.0 ** params["log10_radius"]
    x0 = tensors.initial_ball(center, radius, params["n_traj"])  # (B, 3)
    times, traj = solve.solve(x0, A, Q, params["t_max"])  # (T,), (T, B, 3)
    s = equation.spread(traj)  # (T,)
    fit = solve.lyapunov_fit(times, s)

    print(f"A =\n{A.cpu().numpy()}")
    print(f"non-zero entries of Q: {[tuple(i) for i in Q.nonzero().tolist()]} -> {Q[Q != 0].tolist()}")
    print(
        f"divergence A_ii + (Q_iik + Q_iki) x_k = {equation.divergence(x0[:1], A, Q).item():.4f}"
        f"  (-(σ + 1 + β) = {-(sigma + 1 + beta):.4f})"
    )
    print(f"{traj.shape[1]} trajectories, spread {s[0]:.1e} -> {s[-1]:.2f}")
    print(f"fitted largest Lyapunov exponent λ = {fit[0]:.3f}  (literature: 0.906 for σ=10, ρ=28, β=8/3)")
    return plot.animate(times, traj, s, fit, radius, out_dir / "lorenz_attractor.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/lorenz_attractor/equation.py
r"""Physics → tensor translation.

The Lorenz system (a three-mode model of convection),

    \dot x = \sigma (y - x),\qquad \dot y = x(\rho - z) - y,\qquad \dot z = x y - \beta z,

has a right-hand side that is a linear part plus a quadratic part. For a batch b of trajectories with
state x_{bi} (i = 0, 1, 2 for x, y, z) it reads, in index notation,

    \dot x_{bi} = A_{ij} x_{bj} + Q_{ijk} x_{bj} x_{bk}

with the constant matrix A_{ij} and the rank-3 tensor Q_{ijk} (only two entries are non-zero:
Q_{102} = -1 gives -xz in \dot y, and Q_{201} = +1 gives +xy in \dot z).
Repeated j (and k) are summed: einsum('ij,bj->bi', A, x) and einsum('ijk,bj,bk->bi', Q, x, x).

The divergence of the flow, \partial \dot x_i / \partial x_i = A_{ii} + (Q_{iik} + Q_{iki}) x_k,
is the rate at which a small volume of states grows; for Lorenz it is the constant -(\sigma + 1 + \beta).
"""

import torch


def rhs(x: torch.Tensor, A: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    r"""\dot x_{bi} = A_{ij} x_{bj} + Q_{ijk} x_{bj} x_{bk}.   x: (B, 3), A: (3, 3), Q: (3, 3, 3) → (B, 3)"""
    linear = torch.einsum("ij,bj->bi", A, x)  # sum over j
    quadratic = torch.einsum("ijk,bj,bk->bi", Q, x, x)  # sum over j and k
    return linear + quadratic


def rhs_explicit(x: torch.Tensor, sigma: float, rho: float, beta: float) -> torch.Tensor:
    """The textbook form, component by component (used to check the tensor form).   x: (B, 3) → (B, 3)"""
    X, Y, Z = x.unbind(-1)
    return torch.stack([sigma * (Y - X), X * (rho - Z) - Y, X * Y - beta * Z], dim=-1)


def divergence(x: torch.Tensor, A: torch.Tensor, Q: torch.Tensor) -> torch.Tensor:
    r"""\partial \dot x_i / \partial x_i = A_{ii} + (Q_{iik} + Q_{iki}) x_{bk}.   x: (B, 3) → (B,)"""
    trace_A = torch.einsum("ii->", A)
    return trace_A + torch.einsum("iik,bk->b", Q, x) + torch.einsum("iki,bk->b", Q, x)


def spread(x: torch.Tensor) -> torch.Tensor:
    r"""RMS distance from the mean, s = \sqrt{\langle (x_{bi} - \bar x_i)(x_{bi} - \bar x_i) \rangle_b}.

    x: (T, B, 3) → (T,)
    """
    d = x - x.mean(dim=1, keepdim=True)  # (T, B, 3)
    return torch.einsum("tbi,tbi->t", d, d).div(x.shape[1]).sqrt()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/lorenz_attractor/tensors.py
"""Build the concrete tensors: the linear part A_ij (3, 3), the quadratic part Q_ijk (3, 3, 3)
and a batch of starting points x_bi (B, 3) inside a tiny ball.

float64, because the trajectories start only ~1e-6 apart.
"""

import math

import torch

DTYPE = torch.float64


def lorenz_tensors(sigma: float, rho: float, beta: float, device: torch.device | None = None):
    r"""(A (3, 3), Q (3, 3, 3)) with \dot x_i = A_{ij} x_j + Q_{ijk} x_j x_k."""
    A = torch.tensor(
        [
            [-sigma, sigma, 0.0],  # ẋ = σ(y − x)
            [rho, -1.0, 0.0],  # ẏ = ρx − y  (− xz is quadratic)
            [0.0, 0.0, -beta],  # ż = − βz   (+ xy is quadratic)
        ],
        dtype=DTYPE,
        device=device,
    )
    Q = torch.zeros(3, 3, 3, dtype=DTYPE, device=device)
    Q[1, 0, 2] = -1.0  # ẏ gets − x z
    Q[2, 0, 1] = 1.0  # ż gets + x y
    return A, Q


def fixed_points(sigma: float, rho: float, beta: float, device: torch.device | None = None) -> torch.Tensor:
    """The origin and, for ρ > 1, the two wing centres C± = (±c, ±c, ρ − 1) with c = √(β(ρ − 1)).  → (P, 3)"""
    points = [[0.0, 0.0, 0.0]]
    if rho > 1:
        c = math.sqrt(beta * (rho - 1))
        points += [[c, c, rho - 1], [-c, -c, rho - 1]]
    return torch.tensor(points, dtype=DTYPE, device=device)


def initial_ball(center: torch.Tensor, radius: float, n: int, seed: int = 0) -> torch.Tensor:
    """n starting points at distance `radius` from `center`, in random directions.  center: (3,) → (n, 3)"""
    g = torch.Generator().manual_seed(seed)
    u = torch.randn(n, 3, generator=g, dtype=DTYPE)
    u = u / u.norm(dim=1, keepdim=True)  # unit vectors
    return center + radius * u.to(center.device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/lorenz_attractor/solve.py
"""Integrate the whole batch of trajectories with RK4 and fit the growth rate of their spread."""

import math

import torch

from problems._common.integrators import integrate

from .equation import rhs


def settle(A: torch.Tensor, Q: torch.Tensor, t: float = 10.0, dt: float = 0.005) -> torch.Tensor:
    """Start at (1, 1, 1) and run for a while, so the ball starts on the attractor.  → (3,)"""
    x0 = torch.ones(1, 3, dtype=A.dtype, device=A.device)  # (B=1, 3)
    traj = integrate(lambda x, _t: rhs(x, A, Q), x0, dt, int(round(t / dt)))
    return traj[-1, 0]


def solve(x0: torch.Tensor, A: torch.Tensor, Q: torch.Tensor, t_max: float, dt: float = 0.005):
    """Return times (T,) and trajectories (T, B, 3).  x0: (B, 3)"""
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda x, _t: rhs(x, A, Q), x0, dt, n_steps)
    times = torch.arange(n_steps + 1, dtype=x0.dtype, device=x0.device) * dt
    return times, traj


def lyapunov_fit(times: torch.Tensor, s: torch.Tensor, high: float = 1.0):
    r"""Fit ln s = c + λ t by least squares while the spread grows from 10 s(0) to `high`.

    Past `high` the trajectories are no longer close: the spread saturates at the size of the attractor.
    Returns (λ, t_start, t_end, c); λ is nan if the spread never grows through the window.
    """
    above_low, above_high = (s > 10 * s[0]).nonzero(), (s > high).nonzero()
    if len(above_low) == 0 or len(above_high) == 0 or above_high[0] - above_low[0] < 3:
        return math.nan, math.nan, math.nan, math.nan
    i0, i1 = above_low[0].item(), above_high[0].item()
    t, y = times[i0 : i1 + 1], s[i0 : i1 + 1].log()
    X = torch.stack([torch.ones_like(t), t], dim=1)  # (n, 2) least squares for [c, λ]
    c, lam = torch.linalg.lstsq(X, y[:, None]).solution[:, 0].tolist()
    return lam, t[0].item(), t[-1].item(), c


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/lorenz_attractor/plot.py
"""Left: the butterfly in 3D, with each trajectory's recent path in its own colour. Right: log10 of their spread."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    times: torch.Tensor, traj: torch.Tensor, s: torch.Tensor, fit: tuple, radius: float, path: Path, trail: float = 1.5
) -> Path:
    """traj: (T, B, 3); s: (T,) spread; fit = (λ, t0, t1, c). trail = time units of path behind each point."""
    t = times.cpu().numpy()  # (T,)
    x = traj.cpu().numpy()  # (T, B, 3)
    log_s = s.clamp_min(1e-300).log10().cpu().numpy()  # (T,)
    n_traj = x.shape[1]
    tail = max(2, int(trail / (t[1] - t[0])))
    frames = frame_indices(len(t), 90)

    fig = plt.figure(figsize=(7.6, 4))
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    ax_s = fig.add_subplot(1, 2, 2)

    # --- left: the attractor (grey ghost of trajectory 0) and the coloured trajectories ---
    ghost = x[:: max(1, len(t) // 3000), 0]  # thinned path of one trajectory: the shape of the attractor
    ax.plot(ghost[:, 0], ghost[:, 1], ghost[:, 2], lw=0.4, color="#d1d5db")
    lo, hi = x.min(axis=(0, 1)), x.max(axis=(0, 1))
    ax.set_xlim(lo[0], hi[0])
    ax.set_ylim(lo[1], hi[1])
    ax.set_zlim(lo[2], hi[2])
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.tick_params(labelsize=7)
    colors = [ACCENT[b % len(ACCENT)] for b in range(n_traj)]
    lines = [ax.plot([], [], [], lw=1.2, color=c)[0] for c in colors]
    dots = [ax.plot([], [], [], "o", ms=4, color=c)[0] for c in colors]
    clock = ax.text2D(0.02, 0.95, "", transform=ax.transAxes, fontsize=9)
    ax.view_init(elev=18, azim=-60)
    ax.set_box_aspect(None, zoom=0.88)  # leave room for the z label

    # --- right: how far apart the trajectories are, in powers of ten ---
    ax_s.set_xlim(t[0], t[-1])
    ax_s.set_ylim(log_s.min() - 0.5, log_s.max() + 0.8)
    ax_s.set_xlabel("t")
    ax_s.set_ylabel("log₁₀ spread of the trajectories")
    ax_s.set_title(f"{n_traj} starts within r = {radius:.0e}", fontsize=9)
    trace = ax_s.plot([], [], lw=1.2, color="#111827")[0]
    lam, t0, t1, c = fit
    if lam == lam:  # not nan
        ln10 = 2.302585
        ax_s.plot([t0, t1], [(c + lam * t0) / ln10, (c + lam * t1) / ln10], "--", color="#f59e0b", lw=2)
        ax_s.text(0.03, 0.97, f"fit: λ ≈ {lam:.2f}", transform=ax_s.transAxes, va="top", color="#b45309")

    def update(step: int):
        start = max(0, step - tail)
        for b in range(n_traj):
            seg = x[start : step + 1, b]  # (≤tail, 3) recent path of trajectory b
            lines[b].set_data_3d(seg[:, 0], seg[:, 1], seg[:, 2])
            dots[b].set_data_3d([x[step, b, 0]], [x[step, b, 1]], [x[step, b, 2]])
        clock.set_text(f"t = {t[step]:.1f}")
        trace.set_data(t[: step + 1], log_s[: step + 1])

    fig.tight_layout()
    return save_gif(fig, update, frames, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'sigma': 10.0,
    'rho': 28.0,
    'beta': 2.6667,
    't_max': 30.0,
    'log10_radius': -6,
    'n_traj': 8
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.lorenz_attractor")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.lorenz_attractor.main")
problem_dir = Path("problems/others/lorenz_attractor")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
